# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata  # Metadata is an object, not a dict
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nIdentifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Authors: {metadata.author}")
print(f"Record sets: {getattr(metadata, 'recordSet', [])}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields for reference. As best practice, all references to dataset entities will use their `@id`.

In [ ]:
# List all record sets in the schema with their @id and names
if not getattr(metadata, 'recordSet', []):
    # If not loaded, try refreshing metadata
    record_sets = [rs for rs in dataset._record_sets]
else:
    # Usually the case if the Croissant file was loaded properly
    record_sets = metadata.recordSet

if not record_sets:
    print('No record sets found in the metadata. Please check the dataset URL or the schema structure.')
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}   Name: {rs.get('name', '')}")
        fields = rs.get('field', [])
        print('  Fields:')
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id')}  Name: {field.get('name')}")
            else:
                print(f"    - Field @id: {field}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview for dynamic and reproducible access.

In [ ]:
# For demonstration, we'll automatically extract all available record sets

dataframes = {}

# Prepare a list of record set @ids
record_set_ids = []
for rs in getattr(metadata, 'recordSet', []):
    record_set_ids.append(rs['@id'])

if not record_set_ids:
    # fallback if not in metadata
    record_set_ids = [rs['@id'] for rs in dataset._record_sets]

for record_set_id in record_set_ids:
    # Load records from the dataset using @id reference
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}")
        print(f"Columns: {df.columns.tolist()[:6]} ...\n")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Preview one of the dataframes (first record set)
if record_set_ids:
    print(f"First few rows of record set {record_set_ids[0]}:")
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

Below we demonstrate using field `@id` to select a numeric column and a grouping variable for EDA.

In [ ]:
# For demonstration, select the first record set and inspect numeric fields
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    print(f"Columns for record set {record_set_id}:\n{df.columns.tolist()}")

    # Find a likely numeric field to use by inspecting columns
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break

    # Use the first non-null numeric field
    if numeric_field:
        print(f"Using numeric field @id: {numeric_field} for analysis.\n")
        threshold = df[numeric_field].mean() if df[numeric_field].dtype != object else 0
        # Filter for values above the mean (or above an arbitrary threshold if the mean is NaN)
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by the first categorical field if possible
        group_field = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean of {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No categorical field found to group by.")
    else:
        print("No numeric fields detected in the record set.")
else:
    print("No record sets or records available in the dataset.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using `matplotlib` or `seaborn` for convenience.

In [ ]:
# Simple visualization: plot histogram of the chosen numeric field, if available
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field} in record set {record_set_id}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a grouping field was found, plot boxplots
    if group_field:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this exploration, we loaded the FAIR² Croissant mass spectrometry EVs dataset, reviewed available record sets and fields by `@id`, and demonstrated programmatic analysis and visualization of one record set using `mlcroissant`, pandas, and seaborn. For advanced analytics, repeat the workflow for different record sets and fields.

All data access and operations are performed referencing each entity by their `@id` to ensure reproducibility.